# Suite BLW — A cache layer: a store given a layer below

A store may be given the layer below it, `Builders.Store(below)`, which makes it a cache layer ([CAP-283: a store given a store below](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-283); [the stacked stores proposal, a cache layer and the layer below](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#a-cache-layer-and-the-layer-below), its second stage). Its transaction opens one transaction of the layer below for its whole duration. Each input frame its `exec` takes, it interprets read-only over what it holds, as far as it can know that what it holds is all there is: where it cannot, the frame halts, and goes to the layer below's transaction with the rest of the expression, writing; a frame it completes goes down too. Each frame that comes back is given: one annotated with an error as it is; one complete, its journal applied to what the cache layer holds; one partial, executed again by the cache layer, writing. The cache layer loads nothing on a read: it holds what the journals of the frames that come back, its own writes, and the layer below's follow give it. At its commit, the layer below's transaction commits first, a refusal there refusing both. An install runs below first, then in the cache layer. What another commits below reaches it by the follow, applied to the objects it holds alone ([CAP-285: writes through, others' writes in](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-285)). Given no layer below, a store holds what is written to it, as it always has: the loopback.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import json

from mbse.Schemas.Framework import JSON, Builders as B, Changes, Plain, Provenance, References, Transactions
from mbse.Schemas.Framework.Transactions import Frame
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton crm.TheBook under a key, with a name, an age and a home, who knows persons
    and keeps pets, each owned by its keeper under a name; crm.Note, counted."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Home": S.OfObject, "Note": S.OfObject, "Pet": S.OfObject, "Keeps": S.OfRelation})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]),
            lambda _: _.Home.name("crm.Home").properties({"city": S.OfStr}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}),
            lambda _: _.Keeps.name("crm.Keeps").links([lambda t: t.name("keeper"), lambda t: t.name("pet")]).properties({
                "name": S.OfStr}).uniques([["keeper", "name"], ["pet"]]),
            lambda _: _.Pet.name("crm.Pet").ref(True).adjacencies([
                lambda t: t.name("keeper").relation([_.Keeps]).me("pet").owner(True)]),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt,
                                                                       "home": _.Home}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.name("known_by").relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
                lambda t: t.name("pets").relation([_.Keeps]).me("keeper"),
            ]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
        ])]))])


def person(key, name, age=None, knows=()):
    """A transaction making the contact `key`, named `name`, of `age`, knowing the contacts `knows`."""
    def build(t):
        symbols = {"p": t.crm.Person, "book": t.crm.Book, **{k: t.crm.Person for k in knows}}
        spec = [lambda _: _.book.contacts({key: lambda e: e.contact(_.p), **{
                    k: (lambda e, k=k: e.contact(getattr(_, k))) for k in knows}}),
                lambda _: _.p.name(name) if age is None else _.p.name(name).age(age)]
        spec += [lambda _, k=k: _.p.knows([getattr(_, k)]) for k in knows]
        return t.uid(f"person/{key}").mutations([lambda mu: mu.upsert(lambda u: u.exists([
            lambda p: p.symbols(symbols).spec(spec)]))])
    return build


def seeded(store):
    """`store`, holding crm's schemas, and Ada, 36, who knows Bob, and Bob."""
    store.transaction(crm)
    store.transaction(person("bob", "Bob"))
    store.transaction(lambda t: t.uid("ada/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"a": t.crm.Person, "b": t.crm.Person, "book": t.crm.Book}).spec([
            lambda _: _.book.contacts({"ada": lambda e: e.contact(_.a), "bob": lambda e: e.contact(_.b)}),
            lambda _: _.a.name("Ada").age(36).knows([_.b])])]))]))
    return store


def stacked():
    """A layer below seeded, and a cache layer over it, made after it, holding its schemas."""
    lower = seeded(B.Store())
    cache = B.Store(lower)
    cache.transaction(crm)
    return lower, cache


def paths(store, frames):
    """Each frame's bindings, by symbol, as their paths' JSON, the store's own and another's alike, and its errors."""
    return [({symbol: JSON.dumps(Plain.ToPlain(store)(target.as_schema(), target)["path"])
              for symbol, target in sorted(frame.items())}, [error.kind for error in getattr(frame, "errors", [])]) for frame in frames]


def people(store):
    """Each person the store holds: name, age and whom they know, read within a transaction of its own."""
    with store.transaction():
        return sorted((str(p.name), store.member(p, "age"), sorted(str(e.known.name) for e in p.knows))
                      for p in store.extent("crm.Person"))


def everyone(t):
    """A map of every person."""
    return t.uid("everyone/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": t.crm.Person})]))])


def aged(name, age):
    """A map of the person named `name`, then an update writing their age."""
    return lambda t: t.uid(f"aged/{name}").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
        {"x": t.crm.Person}).spec([lambda _: _.x.name(name)])])).then([lambda w: w.update(lambda u: u.apply([
            lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(age)])]))])])


def frames(items):
    """The frames of a stream's items, its completion markers left out."""
    return [item for item in items if isinstance(item, Frame)]


def contact(store, key):
    """The reference of the contact `key`, a value of `store`'s, by its path."""
    return Plain.FromPlain(store)(store.registered("References.OfObject"), {"path": {
        "singleton": "crm.TheBook", "next": {"entry": {"i": "contacts", "o": "book", "properties": {"key": key},
                                                       "next": {"target": {}}}}}})


def knows(knower, known):
    """A transaction: the person named `knower` knows the one named `known`."""
    return lambda t: t.uid(f"knows/{knower}/{known}").mutations([lambda mu: mu.map(lambda m: m.symbols([
        lambda p: p.symbols({"x": t.crm.Person, "y": t.crm.Person}).spec([
            lambda _: _.x.name(knower), lambda _: _.y.name(known)])])).then([lambda w: w.update(lambda u: u.apply([
                lambda p: p.symbols({"x": t.crm.Person, "y": t.crm.Person}).spec([lambda _: _.x.knows([_.y])])]))])])

<a id="BLW-01"></a>
## BLW-01 · A store given a layer below is a cache layer, `store._below`: an install runs below first, then in it; it loads nothing on a read, holding none of what the layer below held before it, which a visitor of it does not find; a transaction's frames come from below, the same as a store with no layer below gives, each by its reference

In [ ]:
lower, cache = stacked()
twin = seeded(B.Store())
assert cache._below is lower and twin._below is None
assert cache.registered("crm.Person").name == lower.registered("crm.Person").name == "crm.Person"
with cache.transaction():
    book = cache.singleton("crm.TheBook")
    assert list(book.contacts) == [] and list(cache.extent("crm.Person")) == []  # it holds none of it: nothing loaded
given = cache.transaction(everyone)
assert paths(cache, given) == paths(twin, twin.transaction(everyone)) and len(given) == 2
assert people(cache) == []  # a frame given back makes nothing present: its object is the layer below's

<a id="BLW-02"></a>
## BLW-02 · Writes go through: a frame that halts where a term would write goes below with its remainder, and the layer below's transaction writes it; what it made comes back as the frame's journal, applied to what the cache layer holds, within the same transaction, so that an object made through the cache layer is held by it, all of it; the frame given carries the cache layer's journal, under the transaction's uid; what changed only what the cache layer does not hold changes nothing in it

In [ ]:
lower, cache = stacked()
twin = seeded(B.Store())
with cache.transaction() as txn:
    items = list(txn.exec(Frame(), person("cy", "Cy", 30), True))
    (made,) = frames(items)
    assert sorted((str(p.name), cache.member(p, "age")) for p in cache.extent("crm.Person")) == [("Cy", 30)]  # within it
assert items[-1] == {"end": "0"} and made.remainder is None and made.errors == []
(journal,) = Plain.ToPlain(cache)(cache.registered("Changes.Journal"), made.journal)
assert journal["id"] == "person/cy#0" and [list(change) for change in journal["changes"]] == [["entry"], ["property"], ["property"]]
twin.transaction(person("cy", "Cy", 30))
assert people(cache) == [("Cy", 30, [])] and people(lower) == people(twin)  # held above; written below
with cache.transaction():
    assert cache.extent("crm.Person")[0] is cache.singleton("crm.TheBook").contacts({"key": "cy"})
assert cache._knows(cache._present({"singleton": "crm.TheBook", "next": {"entry": {
    "i": "contacts", "o": "book", "properties": {"key": "cy"}, "next": {"target": {}}}}}))  # all of it
older = cache.transaction(aged("Ada", 40))  # Ada, not held above: written below, nothing changed above
assert [frame.journal for frame in older] == [None] and people(cache) == [("Cy", 30, [])]
twin.transaction(aged("Ada", 40))
assert people(lower) == people(twin) and paths(cache, older) == paths(twin, twin.transaction(lambda t: t.mutations([
    lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.name("Ada")])]))])))
homed = lambda t: t.uid("homed/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": contact(cache, "cy")})])).then([lambda w: w.update(lambda u: u.apply([lambda p: p.symbols(
        {"x": contact(cache, "cy")}).spec([lambda _: _.x.home(lambda h: h.city("Oslo"))])]))])])
(home,) = cache.transaction(homed)  # a value object it assigns, of the layer below's, given as the cache layer's
assert Plain.ToPlain(cache)(cache.registered("crm.Home"), home.assigns["x"]["home"]) == {"city": "Oslo"}
nobody = lambda t: t.uid("nobody/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": contact(cache, "cy")})])).then([lambda w: w.update(lambda u: u.apply([lambda p: p.symbols(
        {"x": contact(cache, "cy")}).spec([lambda _: _.x.age(31)])]).then([lambda w2: w2.map(lambda m: m.symbols([
            lambda p: p.symbols({"x": t.crm.Person, "y": t.crm.Person}).spec([lambda _: _.y.name("Zed")])]))]))])])
with cache.transaction() as txn:  # no frame given, below: what it changed comes back on its completion marker
    items = list(txn.exec(Frame(), nobody, True))
    assert frames(items) == [] and [(str(p.name), cache.member(p, "age")) for p in cache.extent("crm.Person")] == [
        ("Cy", 31)]  # applied within it
(marked,) = Plain.ToPlain(cache)(cache.registered("Changes.Journal"), items[-1]["journal"])
assert items[-1]["end"] == "0" and marked["changes"][0]["property"]["name"] == "age"

<a id="BLW-03"></a>
## BLW-03 · Errors are forwarded: a frame the layer below annotates with an error is given as it is, its errors as the layer below's interpreter wrote them; the scope then raises the first, and neither layer commits anything

In [ ]:
lower, cache = stacked()
twin = seeded(B.Store())
Person = lower.registered("crm.Person")
with lower.transaction():  # a spec naming Bob by his path, built while he is held
    knows_bob = Plain.ToPlain(lower).Transaction(lower.Mutations.OfTransaction(lambda t: t.uid("knows/1").mutations([
        lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": Person}).spec([
            lambda _: _.x.name("Ada")])])).then([lambda w: w.update(lambda u: u.apply([lambda p: p.symbols(
                {"x": Person}).spec([lambda _: _.x.age(1).knows([lower.singleton("crm.TheBook").contacts(
                    {"key": "bob"})])])]))])])))
for store in (lower, twin):
    with store.transaction():
        store.crm.Book(store.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
seen = []
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    with cache.transaction() as txn:
        seen += list(txn.exec(Frame(), Plain.FromPlain(cache).Transaction(knows_bob), True))
(failed,) = frames(seen)
assert [(e.subterm, e.kind, e.message) for e in failed.errors] == [
    ("$.mutations[0].map.then[0].update", "LookupError", "no object of the store has the path a spec refers to")]
assert paths(cache, [failed])[0][0] == {"x": JSON.dumps({"singleton": "crm.TheBook", "next": {"entry": {
    "i": "contacts", "o": "book", "properties": {"key": "ada"}, "next": {"target": {}}}}})}
assert people(lower) == [("Ada", 36, [])] and people(cache) == []  # nothing committed in either
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    with cache.transaction() as txn:
        (again,) = frames(txn.exec(failed, Plain.FromPlain(cache).Transaction(knows_bob), True))  # annotated: as it is
assert again.errors == failed.errors and again.remainder is None

<a id="BLW-04"></a>
## BLW-04 · The layer below commits first: the cache layer's transaction holds one transaction of the layer below open for its whole duration, which commits within the cache layer's commit, before it; a refusal below, a unique clause broken against an entry the cache layer does not hold, refuses both, and where the cache layer's raises, the layer below's is dropped

In [ ]:
lower, cache = stacked()
order = []
Changes.observe(lower, lambda journal: order.append(("below", cache._proposals is not None)))
Changes.observe(cache, lambda journal: order.append(("cache", cache._proposals is not None)))
with cache.transaction() as txn:
    list(txn.exec(Frame(), person("cy", "Cy"), True))
    assert lower._proposals is not None  # the layer below's transaction, open as long as the cache layer's
assert order == [("below", True), ("cache", False)]  # the layer below committed within the cache layer's commit
with raises(ValueError, match="breaks unique"):  # a second contact keyed "ada", which the cache layer does not hold
    cache.transaction(person("ada", "Eve"))
assert people(lower) == [("Ada", 36, ["Bob"]), ("Bob", None, []), ("Cy", None, [])] and people(cache) == [("Cy", None, [])]
with raises(ValueError, match="stop"):
    with cache.transaction() as txn:
        list(txn.exec(Frame(), person("dee", "Dee"), True))
        raise ValueError("stop")  # the cache layer's raises: the layer below's is dropped
assert lower._proposals is None and [name for name, *_ in people(lower)] == ["Ada", "Bob", "Cy"]
assert people(cache) == [("Cy", None, [])]

<a id="BLW-05"></a>
## BLW-05 · Given no layer below, a store holds what is written to it: its `exec`, read-only, halts a frame that would write, which the same transaction executes again, writing, the loopback; nothing is forwarded, and its frames carry its own journals

In [ ]:
alone = seeded(B.Store())
assert alone._below is None and alone._lower is None
with alone.transaction() as txn:
    assert alone._lower is None
    (halted,) = frames(txn.exec(Frame(), aged("Ada", 37), False))
    (looped,) = frames(txn.exec(halted, halted.remainder, True))  # the loopback
assert halted.remainder.subterm == "$.mutations[0].map.then[0].update" and looped.remainder is None
(journal,) = Plain.ToPlain(alone)(alone.registered("Changes.Journal"), looped.journal)
assert journal["changes"] == [{"property": {"holder": "o0", "name": "age", "mutations": [{"set": {"value": 37}}]}}]
assert people(alone)[0] == ("Ada", 37, ["Bob"])

<a id="BLW-06"></a>
## BLW-06 · A partial frame that comes back from a real layer below, as well as from the loopback, is executed again by the cache layer, writing, and its writes are sent below as it makes them: a layer below that gives frames back unfinished, as one interpreting read-only does, still has them written

In [ ]:
class Halting(B.Store):
    """A layer below whose transaction's `exec` interprets the first frame it is given read-only, whatever it is asked,
    as a layer that gives frames back unfinished does, once `armed`: what it gives back partial, kept in `partial`."""

    armed = False
    partial: list = []

    def _scope(self):
        scope = super()._scope()

        class Scope:
            def __enter__(_):
                txn = scope.__enter__()
                given, asked = txn.exec, []

                def exec(frame, remainder, allow_write):
                    asked.append(remainder)
                    for item in given(frame, remainder, allow_write and (len(asked) > 1 or not self.armed)):
                        if isinstance(item, Frame) and item.remainder is not None:
                            self.partial.append(item)
                        yield item
                txn.exec = exec
                return txn

            def __exit__(_, *raised):
                return scope.__exit__(*raised)
        return Scope()


lower = seeded(Halting())
cache = B.Store(lower)
cache.transaction(crm)
lower.armed = True
given = cache.transaction(person("cy", "Cy", 7))  # the upsert comes back partial; the cache layer writes it, through
assert [frame.remainder.subterm for frame in lower.partial] == ["$.mutations[0].upsert"]
assert [(frame.remainder, frame.errors) for frame in given] == [(None, [])]
assert people(cache) == [("Cy", 7, [])] and ("Cy", 7, []) in people(lower)

<a id="BLW-07"></a>
## BLW-07 · Others' writes in: what another commits below, the layer below's follow gives the cache layer as a one-transaction journal, which it applies to what it holds alone, in the order the layer below committed: an object made under one it holds is made in it, and one made under one it does not hold is not; a property of one it holds is set; what it does not hold changes nothing; an entry linking one it does not hold is left out, and the objects it holds that the entry links it holds not all of from then; its followers are told what changed, under the id it arrived with

In [ ]:
lower, cache = stacked()
journals = []
Changes.observe(cache, journals.append)
lower.transaction(person("cy", "Cy"))  # made by another below: made above, under the book it holds
lower.transaction(aged("Cy", 5))  # a property of one it holds: set
lower.transaction(aged("Ada", 50))  # one it does not hold: nothing
lower.transaction(lambda t: t.uid("rex/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person}).spec([lambda _: _.x.name("Ada")])])).then([lambda w: w.upsert(lambda u: u.exists([
        lambda p: p.symbols({"x": t.crm.Person, "r": t.crm.Pet}).spec([
            lambda _: _.x.pets({"rex": lambda e: e.pet(_.r)})])]))])]))  # made under one it does not hold: nothing
assert people(cache) == [("Cy", 5, [])] and len(journals) == 2
assert [transaction["id"] for journal in journals
        for transaction in Plain.ToPlain(cache)(cache.registered("Changes.Journal"), journal)] == ["person/cy", "aged/Cy"]
with cache.transaction():
    cy = cache.singleton("crm.TheBook").contacts({"key": "cy"})
    assert cache._knows(cy)
lower.transaction(knows("Cy", "Ada"))  # an entry linking Ada, not held above: left out, and Cy held not all of
with cache.transaction():
    cy = cache.singleton("crm.TheBook").contacts({"key": "cy"})
    assert list(cy.knows) == [] and not cache._knows(cy)
assert len(journals) == 2 and ("Cy", 5, ["Ada"]) in people(lower)

<a id="BLW-08"></a>
## BLW-08 · Layers stack: a cache layer over a cache layer over a store; a transaction goes down to the lowest layer and its frames come back up, each layer applying their journals to what it holds; what another writes to the lowest reaches each, through each follow

In [ ]:
lower, middle = stacked()
top = B.Store(middle)
top.transaction(crm)
given = top.transaction(person("cy", "Cy", 3))
assert people(top) == people(middle) == [("Cy", 3, [])] and ("Cy", 3, []) in people(lower)
assert given[0].journal is not None and given[0].remainder is None
twin = seeded(B.Store())
twin.transaction(person("cy", "Cy", 3))
assert paths(top, top.transaction(everyone)) == paths(twin, twin.transaction(everyone))
lower.transaction(aged("Cy", 4))  # another's write, to the lowest
assert people(top) == people(middle) == [("Cy", 4, [])]

<a id="BLW-09"></a>
## BLW-09 · The completeness rule: a cache layer interpreting read-only binds and reads only what it can know it holds all of; elsewhere the frame halts, and the layer below binds it. A symbol of a schema halts, the cache layer holding no whole extent; a singleton binds, but its spec halts, the cache layer holding not all of it; a reference binds where the cache layer holds its object, and its spec is read there where it holds all of it, halting where it holds not all of it, or none; an update or an upsert halts. Whatever halts, a cache layer gives the frames a store with no layer below holding the same gives, never fewer

In [ ]:
lower, cache = stacked()
twin = seeded(B.Store())
for store in (cache, twin):
    store.transaction(person("cy", "Cy", 9))  # Cy, made through the cache layer: it holds all of her
lower.transaction(person("dan", "Dan"))  # Dan, made below by another, under the book it holds: held all of too
twin.transaction(person("dan", "Dan"))
lower.transaction(knows("Dan", "Ada"))  # Dan knows Ada, not held: Dan held not all of from then
twin.transaction(knows("Dan", "Ada"))


def named(store, key, name):
    """A map binding x to the contact `key` by its reference, whose spec gives its name."""
    return lambda t: t.uid(f"named/{key}").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
        {"x": contact(store, key)}).spec([lambda _: _.x.name(name)])]))])


def renamed(store, key, name):
    """`named`, then an update writing the name it gave."""
    return lambda t: t.uid(f"renamed/{key}").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
        {"x": contact(store, key)}).spec([lambda _: _.x.name(name)])])).then([lambda w: w.update(lambda u: u.apply([
            lambda p: p.symbols({"x": contact(store, key)}).spec([lambda _: _.x.name(name)])]))])])


def book(t):
    """A map binding b to the book, then one whose spec reads its contacts."""
    return t.uid("book/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
        {"b": t.crm.Book})])).then([lambda w: w.map(lambda m: m.symbols([lambda p: p.symbols(
            {"b": t.crm.Book, "c": t.crm.Person}).spec([lambda _: _.b.contacts([_.c])])]))])])


cases = {  # each expression, and where read-only the cache layer halts it, None where it completes it
    "a schema's objects": (lambda store: everyone, "$.mutations[0].map"),
    "a singleton, its spec": (lambda store: book, "$.mutations[0].map.then[0].map"),
    "a reference held all of": (lambda store: named(store, "cy", "Cy"), None),
    "a reference held all of, its spec not holding": (lambda store: named(store, "cy", "Zoe"), None),
    "a reference held not all of": (lambda store: named(store, "dan", "Dan"), "$.mutations[0].map"),
    "a reference not held": (lambda store: named(store, "ada", "Ada"), "$.mutations[0].map"),
    "an update, of what it holds all of": (lambda store: renamed(store, "cy", "Cy"), "$.mutations[0].map.then[0].update"),
    "an upsert": (lambda store: person("cy", "Cy", 9), "$.mutations[0].upsert"),
}
for case, (expression, halts) in cases.items():
    with cache.transaction() as txn:
        local = frames(txn.exec(Frame(), expression(cache), False))
    assert [frame.remainder and frame.remainder.subterm for frame in local] == ([] if case.endswith("not holding") else [halts]), case
    written, alone = cache.transaction(expression(cache)), twin.transaction(expression(twin))
    assert paths(cache, written) == paths(twin, alone), case  # the same frames: never fewer
assert people(lower) == people(twin)
given = {"x": contact(cache, "ada")}  # an input frame binding a reference the cache layer does not hold
dropping = lambda store: lambda t: t.uid("dropping/1").mutations([lambda mu: mu.reduce(lambda r: r.symbols([
    lambda c: c.symbols(["x"])]))])
for expression, halts in ((lambda store: named(store, "ada", "Ada"), "$.mutations[0].map"),
                          (dropping, "$.mutations[0].reduce")):
    with cache.transaction() as txn:
        (local,) = frames(txn.exec(given, expression(cache), False))  # kept as it is, unresolved: it halts there
    assert local.remainder.subterm == halts and paths(cache, [local])[0][0] == paths(cache, [given])[0][0]
    with cache.transaction() as txn:
        written = frames(txn.exec(given, expression(cache), True))
    with twin.transaction() as txn:
        alone = frames(txn.exec({"x": contact(twin, "ada")}, expression(twin), True))
    assert paths(cache, written) == paths(twin, alone)

<a id="BLW-10"></a>
## BLW-10 · A cache layer's own writes, by builders in its transaction, go through as they are made, the layer below's transaction executing each, so that the layer below holds them within the transaction; an object made without its owner is refused; a transaction is recorded once, in the lowest layer, which the cache layer sends its uid; what the layer below reports of it as it commits is taken into the cache layer's, whose followers are given one journal of it, the record among it

In [ ]:
lower, cache = stacked()
cache.transaction(Provenance.install)  # installs run below first, then in the cache layer
cache.transaction(lambda t: Provenance.extend(t, [t.crm.Person]))
assert lower.registered("Provenance.CreatedBy").name == "Provenance.CreatedBy"
journals = []
Changes.observe(cache, journals.append)
with cache.transaction():
    cache.crm.Person(lambda p: p.book(lambda a: a.insert(lambda e: e.key("gus").book(cache.singleton("crm.TheBook"))))
                     .name("Gus"))
    assert lower.singleton("crm.TheBook").contacts({"key": "gus"}).name.get() == "Gus"  # below, within it
    uid = Transactions._transaction_uid(cache)
with raises(ValueError, match="give it one in the same write"):
    with cache.transaction():
        cache.crm.Person(lambda p: p.name("Nobody"))  # no owner: it cannot go through as it is made
with lower.transaction():
    recorded = [str(e.uid) for e in lower.singleton("Provenance.Transactions").transactions]
with cache.transaction():
    held = [str(e.uid) for e in cache.singleton("Provenance.Transactions").transactions]
(journal,) = journals
(transaction,) = Plain.ToPlain(cache)(cache.registered("Changes.Journal"), journal)
assert recorded == held == [uid] and transaction["id"] == uid  # recorded once, below; taken into the cache layer's
adjacencies = {change["entry"]["adjacency"] for change in transaction["changes"] if "entry" in change}
assert {"contacts", "created", "transactions"} <= adjacencies  # its write, and the record the lowest made of it
assert people(cache) == [("Gus", None, [])] and ("Gus", None, []) in people(lower)

<a id="BLW-11"></a>
## BLW-11 · Split-join: a frame that halts within a branch of a partition goes below with the rest of that branch alone; the cache layer, which split it, collects what each branch ends with, joins it, and goes on with the terms after the join; the frames are those a store with no layer below gives

In [ ]:
lower, cache = stacked()
twin = seeded(B.Store())
for store in (cache, twin):
    store.transaction(person("cy", "Cy", 9))  # held all of by the cache layer


def split(store):
    """Cy, by her reference, then a partition: its first branch ages her, its second keeps her as she is; then the
    book."""
    arm = lambda then: lambda e: e.branch(lambda b: b.then([then]))
    cy = contact(store, "cy")
    return lambda t: t.uid("split/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
        {"x": cy})])).then([lambda w: w.partition(lambda q: q.branches([
            arm(lambda w2: w2.update(lambda u: u.apply([lambda p: p.symbols({"x": cy}).spec([lambda _: _.x.age(10)])]))),
            arm(lambda w2: w2.map(lambda m2: m2.symbols([lambda p: p.symbols({"x": cy}).spec([lambda _: _.x.name("Cy")])]))),
        ]).then([lambda w3: w3.map(lambda m3: m3.symbols([lambda p: p.symbols({"bk": t.crm.Book})]))]))])])


sent = []
original = Transactions._Lower.forward
Transactions._Lower.forward = lambda self, frame, rest: sent.append(rest.subterm) or original(self, frame, rest)
try:
    given = cache.transaction(split(cache))
finally:
    Transactions._Lower.forward = original
assert sent == ["$.mutations[0].map.then[0].partition.branches[0].then[0].update", "$", "$"]  # a branch's rest; then
# each frame joined and gone on with, complete, with nothing left to execute
assert paths(cache, given) == paths(twin, twin.transaction(split(twin))) and len(given) == 2
assert all("bk" in frame for frame in given) and people(lower) == people(twin) and people(cache) == [("Cy", 10, [])]

<a id="BLW-12"></a>
## BLW-12 · What cannot pass between layers is refused, not lost: a counted object, which has no reference, binds a frame the layer below gives back refused, as any frame given back binds one; given to the cache layer to send below, the frame is annotated; a schema object the cache layer does not hold, bound below, binds the frame it comes back in refused

In [ ]:
lower, cache = stacked()
note = lambda t: t.uid("note/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"n": t.crm.Note}).spec([lambda _: _.n.text("hi")])]))])
retext = lambda t: t.uid("retext/1").mutations([lambda mu: mu.update(lambda u: u.apply([lambda p: p.symbols(
    {"n": t.crm.Note}).spec([lambda _: _.n.text("ho")])]))])
with raises(LookupError, match="has no reference"):
    with cache.transaction() as txn:
        (noted,) = frames(txn.exec(Frame(), note, True))  # made below: it comes back refused
        with raises(LookupError, match="symbol 'n' of the frame: a counted 'crm.Note' has no reference"):
            noted["n"]
        (sent,) = frames(txn.exec(noted, retext, True))  # refused here, as any frame given back binds one
assert sent.errors[0].message.startswith("symbol 'n' of the frame: a counted 'crm.Note' has no reference")
lower.transaction(lambda t: t.uid("lab/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"Rack": t.Schemas.OfObject}).spec([lambda _: _.Rack.name("lab.Rack").ref(True)])]))]))  # below alone
racks = lambda t: t.uid("racks/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"s": t.Schemas.OfObject}).spec([lambda _: _.s.name("lab.Rack")])]))])
(rack,) = cache.transaction(racks)
with raises(LookupError, match="symbol 's' of the frame: a schema object the layer it goes to does not hold"):
    rack["s"]